# spharmgrid — ERA5 850-hPa spherical harmonic explorer

Compare ERA5 850-hPa wind before and after vector spherical-harmonic filtering or fixed-grid regridding. The notebook runs in JupyterLab and with `panel serve`.

In [ ]:
from functools import lru_cache
from pathlib import Path

import cartopy.crs as ccrs
import geoviews as gv
import holoviews as hv
import numpy as np
import panel as pn
import pooch
import xarray as xr

import spharmgrid as sg

pn.extension()
hv.extension("bokeh")

DATA_RELEASE = "v0.2.0-data"
DATA_BASE = f"https://github.com/mwyau/PyStormTracker-Data/releases/download/{DATA_RELEASE}/"
UV_FILE = "era5_uv850_2025-2026_djf_2.5x2.5.nc"
VO_FILE = "era5_vo850_2025-2026_djf_2.5x2.5.nc"
REGISTRY = {
    UV_FILE: "sha256:43cbc346a52c5230ac34eb22c7a640800fbffad40da4058686c8042a76bc5965",
    VO_FILE: "sha256:46ce78cd3b065d3777c2d628cdc2311d68a9fcb4d3a3b9948db7c7376ae7a6aa",
}
DATA = pooch.create(
    path=Path(pooch.os_cache("spharmgrid")) / "interactive-v0.2.0-data",
    base_url=DATA_BASE,
    registry=REGISTRY,
)


def fetch(name: str) -> Path:
    return Path(DATA.fetch(name, progressbar=False))


def pressure_hpa(coord: xr.DataArray) -> float:
    value = float(coord.values[0])
    units = str(coord.attrs.get("units", "")).strip().lower()
    if units in {"pa", "pascal", "pascals"}:
        return value / 100.0
    if units in {"hpa", "hectopascal", "hectopascals", "mbar", "millibar", "millibars"}:
        return value
    if not units and np.isclose(value, 850.0):
        return value
    if not units and np.isclose(value, 85000.0):
        return value / 100.0
    raise ValueError(f"Unsupported pressure units {coord.attrs.get('units')!r}")


with xr.open_dataset(fetch(UV_FILE), engine="h5netcdf") as raw:
    if {"u", "v"} - set(raw.data_vars):
        raise ValueError("Pinned ERA5 file must contain u and v")
    if raw.sizes.get("pressure_level") != 1:
        raise ValueError("Expected one pressure level")
    if not np.isclose(pressure_hpa(raw["pressure_level"]), 850.0):
        raise ValueError("Expected the 850-hPa pressure level")
    ERA5 = raw[["u", "v"]].isel(pressure_level=0, drop=True).load()

TIMES = ERA5["valid_time"].values
source_grid = sg.detect_grid(ERA5["u"].isel(valid_time=0))
if source_grid.kind != "cc":
    raise ValueError(f"Expected CC grid, found {source_grid.kind!r}")


def triangular_limit(grid: sg.Grid) -> int:
    latitude_lmax = grid.nlat - 2 if grid.kind == "cc" else grid.nlat - 1
    return min(latitude_lmax, (grid.nlon - 1) // 2)


SOURCE_LMAX = triangular_limit(source_grid)
if SOURCE_LMAX < 42:
    raise ValueError(f"Pinned grid supports only T{SOURCE_LMAX}")

latitude_order = "descending" if source_grid.latitude[0] > source_grid.latitude[-1] else "ascending"
gl_target = sg.gaussian_grid(
    source_grid.nlat - 1,
    source_grid.nlon,
    lon0=float(source_grid.longitude[0]),
    latitude_order=latitude_order,
)

frame0 = ERA5.isel(valid_time=0)
{
    "grid": frame0.sg.grid_type,
    "shape": ERA5["u"].shape,
    "spectral_limit": f"T{SOURCE_LMAX}",
    "kinematics": list(frame0.sg.kinematics().data_vars),
    "potentials": list(frame0.sg.potentials().data_vars),
}

## Processing

`ERA5 CC` filtering calls `sg.regrid_vector` with the source grid as the target, so `u` and `v` are transformed together. `Gauss–Legendre` applies the same spectral selection while synthesizing to a 72×144 GL grid.

In [ ]:
DIAGNOSTICS = (
    "Wind",
    "Relative vorticity",
    "Divergence",
    "Streamfunction",
    "Velocity potential",
    "Rotational wind",
    "Divergent wind",
)
GRID_MODES = ("ERA5 CC", "Gauss–Legendre")


@lru_cache(maxsize=72)
def frame(index: int) -> tuple[xr.DataArray, xr.DataArray]:
    current = ERA5.isel(valid_time=int(index))
    return current["u"], current["v"]


@lru_cache(maxsize=36)
def processed_wind(index: int, lmin: int, lmax: int, taper: float | None, grid_name: str) -> xr.Dataset:
    target = source_grid if grid_name == "ERA5 CC" else gl_target
    max_degree = min(SOURCE_LMAX, triangular_limit(target))
    if not 0 <= lmin <= lmax <= max_degree:
        raise ValueError(f"Invalid spectral range T{lmin}-{lmax}")
    u, v = frame(index)
    return sg.regrid_vector(u, v, target, lmin=lmin, lmax=lmax, taper=taper)


def diagnostic_state(u: xr.DataArray, v: xr.DataArray, diagnostic: str):
    if diagnostic == "Wind":
        return "wind", np.hypot(u, v), u, v, 1.0, "Wind speed (m s⁻¹)"

    if diagnostic in {"Relative vorticity", "Divergence"}:
        kin = sg.kinematics(u, v)
        if diagnostic == "Relative vorticity":
            return "scalar", kin["vo"], None, None, 1e5, "Relative vorticity (10⁻⁵ s⁻¹)"
        return "scalar", kin["d"], None, None, 1e5, "Divergence (10⁻⁵ s⁻¹)"

    if diagnostic in {"Streamfunction", "Velocity potential"}:
        pot = sg.potentials(u, v)
        if diagnostic == "Streamfunction":
            return "scalar", pot["strf"], None, None, 1e-6, "Streamfunction (10⁶ m² s⁻¹)"
        return "scalar", pot["vp"], None, None, 1e-6, "Velocity potential (10⁶ m² s⁻¹)"

    kin = sg.kinematics(u, v)
    if diagnostic == "Rotational wind":
        wind = sg.rotational_wind(kin["vo"], quantity="vorticity")
        return "wind", np.hypot(wind["u_rotational"], wind["v_rotational"]), wind["u_rotational"], wind["v_rotational"], 1.0, "Rotational wind (m s⁻¹)"

    wind = sg.divergent_wind(kin["d"], quantity="divergence")
    return "wind", np.hypot(wind["u_divergent"], wind["v_divergent"]), wind["u_divergent"], wind["v_divergent"], 1.0, "Divergent wind (m s⁻¹)"


@lru_cache(maxsize=72)
def original_state(index: int, diagnostic: str):
    return diagnostic_state(*frame(index), diagnostic)


@lru_cache(maxsize=72)
def processed_state(index: int, lmin: int, lmax: int, taper: float | None, grid_name: str, diagnostic: str):
    wind = processed_wind(index, lmin, lmax, taper, grid_name)
    return diagnostic_state(wind["u"], wind["v"], diagnostic)

In [ ]:
MAP_CRS = ccrs.PlateCarree()
COASTLINE = gv.feature.coastline()
VECTOR_STRIDE = 4


def values(field: xr.DataArray) -> np.ndarray:
    return np.asarray(field.values, dtype=float)


def plotting_array(field: xr.DataArray):
    field = field.transpose("latitude", "longitude")
    longitude = (np.asarray(field["longitude"]) + 180.0) % 360.0 - 180.0
    order = np.argsort(longitude)
    return longitude[order], np.asarray(field["latitude"]), values(field)[:, order]


def plotting_wind(u: xr.DataArray, v: xr.DataArray):
    longitude, latitude, u_values = plotting_array(u)
    _, _, v_values = plotting_array(v)
    return longitude, latitude, u_values, v_values


def common_limits(original, processed):
    data = np.concatenate((values(original[1]).ravel() * original[4], values(processed[1]).ravel() * processed[4]))
    data = data[np.isfinite(data)]
    if original[0] == "scalar":
        limit = max(float(np.percentile(np.abs(data), 99.0)), 1e-12)
        return -limit, limit
    return 0.0, max(float(np.percentile(data, 99.0)), 1.0)


def map_options(title: str):
    return dict(
        width=620,
        height=360,
        xlim=(-180, 180),
        ylim=(-90, 90),
        projection=MAP_CRS,
        title=title,
        tools=["hover", "pan", "wheel_zoom", "reset"],
    )


def draw(state, clim, title: str):
    kind, field, u, v, scale, label = state
    longitude, latitude, data = plotting_array(field)
    mesh = gv.QuadMesh(
        (longitude, latitude, data * scale),
        kdims=["longitude", "latitude"],
        vdims=[label],
        crs=MAP_CRS,
    ).opts(
        cmap="RdBu_r" if kind == "scalar" else "Viridis",
        colorbar=True,
        clim=clim,
        **map_options(title),
    )
    coast = COASTLINE.opts(line_color="#374151", line_width=0.8, projection=MAP_CRS)
    if kind == "scalar":
        return mesh * coast

    longitude, latitude, u_values, v_values = plotting_wind(u, v)
    dl = longitude[::VECTOR_STRIDE]
    dy = latitude[1:-1:VECTOR_STRIDE]
    du = u_values[1:-1:VECTOR_STRIDE, ::VECTOR_STRIDE]
    dv = v_values[1:-1:VECTOR_STRIDE, ::VECTOR_STRIDE]
    vectors = gv.VectorField(
        (dl, dy, np.arctan2(dv, du), np.hypot(du, dv)),
        kdims=["longitude", "latitude"],
        vdims=["angle", "magnitude"],
        crs=MAP_CRS,
    ).opts(
        color="#111827",
        line_width=1,
        pivot="mid",
        projection=MAP_CRS,
        xlim=(-180, 180),
        ylim=(-90, 90),
    )
    return mesh * vectors * coast


def render(side, index, spectral_range, taper_enabled, taper_value, grid_name, diagnostic):
    lmin, lmax = map(int, spectral_range)
    taper = float(taper_value) if taper_enabled else None
    original = original_state(int(index), diagnostic)
    processed = processed_state(int(index), lmin, lmax, taper, grid_name, diagnostic)
    state = original if side == "original" else processed
    return draw(state, common_limits(original, processed), f"{side.title()} · {diagnostic}")


def timestamp(index: int) -> str:
    return np.datetime_as_string(TIMES[int(index)], unit="m").replace("T", " ") + " UTC"

## Interactive application

The expensive spectral controls use throttled values. The notebook uses a normal Panel layout rather than `FastListTemplate`; embedding that full-page template in JupyterLab caused the oversized four-corner/fullscreen glyph.

In [ ]:
diagnostic_widget = pn.widgets.Select(name="Diagnostic", options=list(DIAGNOSTICS), value="Wind")
time_player = pn.widgets.Player(name="Time", start=0, end=len(TIMES) - 1, value=0, interval=1800, loop_policy="loop", show_value=False)
spectral_widget = pn.widgets.IntRangeSlider(name="Spectral degree range", start=0, end=SOURCE_LMAX, value=(0, 42), step=1)
taper_enabled = pn.widgets.Checkbox(name="Taper enabled", value=False)
taper_value = pn.widgets.FloatSlider(name="Taper endpoint response", start=0.01, end=1.0, step=0.01, value=0.1, disabled=True)
grid_widget = pn.widgets.Select(name="Output grid", options=list(GRID_MODES), value="ERA5 CC")

def toggle_taper(event):
    taper_value.disabled = not bool(event.new)

taper_enabled.param.watch(toggle_taper, "value")

bindings = dict(
    index=time_player.param.value,
    spectral_range=spectral_widget.param.value_throttled,
    taper_enabled=taper_enabled.param.value,
    taper_value=taper_value.param.value_throttled,
    grid_name=grid_widget.param.value,
    diagnostic=diagnostic_widget.param.value,
)
left_plot = hv.DynamicMap(pn.bind(render, side="original", **bindings), kdims=[])
right_plot = hv.DynamicMap(pn.bind(render, side="processed", **bindings), kdims=[])


def summary(index, spectral_range, enabled, endpoint, grid_name):
    lo, hi = map(int, spectral_range)
    filtering = f"taper endpoint {endpoint:g}" if enabled else "hard selection"
    return f"**{timestamp(index)}** · `T{lo}–{hi}` · {filtering} · `{grid_name}`"

summary_pane = pn.pane.Markdown(
    pn.bind(
        summary,
        time_player.param.value,
        spectral_widget.param.value_throttled,
        taper_enabled.param.value,
        taper_value.param.value_throttled,
        grid_widget.param.value,
    )
)

vo_reference = None

def relative_rms(reference, candidate):
    a, b = values(reference), values(candidate)
    denominator = np.sqrt(np.nanmean(a * a))
    return float(np.sqrt(np.nanmean((a - b) ** 2)) / denominator) if denominator else 0.0


def vector_relative_rms(au, av, bu, bv):
    return max(relative_rms(au, bu), relative_rms(av, bv))


def run_checks(_):
    global vo_reference
    lmin, lmax = map(int, spectral_widget.value_throttled)
    taper = float(taper_value.value_throttled) if taper_enabled.value else None
    wind = processed_wind(time_player.value, lmin, lmax, taper, grid_widget.value)
    kin = sg.kinematics(wind["u"], wind["v"])
    pot = sg.potentials(wind["u"], wind["v"])
    helm = sg.helmholtz(wind["u"], wind["v"])
    restored = sg.wind(kin["vo"], kin["d"], source="vorticity_divergence")
    grad = sg.gradient(pot["vp"])
    inv_grad = sg.inverse_gradient(grad["gradient_eastward"], grad["gradient_northward"])
    rot = sg.rotational_wind(kin["vo"], quantity="vorticity")
    div = sg.divergent_wind(kin["d"], quantity="divergence")
    vector_lap = sg.vector_laplacian(wind["u"], wind["v"])
    vector_roundtrip = sg.inverse_vector_laplacian(vector_lap["u"], vector_lap["v"])

    metrics = [
        ("wind reconstruction", vector_relative_rms(wind["u"], wind["v"], restored["u"], restored["v"])),
        ("Helmholtz sum", vector_relative_rms(wind["u"], wind["v"], helm["u_rotational"] + helm["u_divergent"], helm["v_rotational"] + helm["v_divergent"])),
        ("laplacian(strf) vs vo", relative_rms(kin["vo"], sg.laplacian(pot["strf"]))),
        ("laplacian(vp) vs d", relative_rms(kin["d"], sg.laplacian(pot["vp"]))),
        ("inverse_laplacian(vo) vs strf", relative_rms(pot["strf"], sg.inverse_laplacian(kin["vo"]))),
        ("inverse_gradient(gradient(vp)) vs vp", relative_rms(pot["vp"], inv_grad)),
        ("rotational + divergent wind", vector_relative_rms(wind["u"], wind["v"], rot["u_rotational"] + div["u_divergent"], rot["v_rotational"] + div["v_divergent"])),
        ("vector Laplacian round trip", vector_relative_rms(wind["u"], wind["v"], vector_roundtrip["u"], vector_roundtrip["v"])),
    ]
    lines = ["### Consistency checks"] + [f"{name}: `{value:.3e}`" for name, value in metrics]

    try:
        if vo_reference is None:
            with xr.open_dataset(fetch(VO_FILE), engine="h5netcdf") as raw:
                vo_reference = raw["vo"].isel(pressure_level=0, drop=True).load()
        reference = vo_reference.isel(valid_time=int(time_player.value))
        calculated = sg.vorticity(*frame(time_player.value))
        a, b = values(reference).ravel(), values(calculated).ravel()
        valid = np.isfinite(a) & np.isfinite(b)
        lines.append(f"ERA5 vo correlation: `{np.corrcoef(a[valid], b[valid])[0, 1]:.6f}`")
    except Exception as exc:
        lines.append(f"External ERA5 vo comparison unavailable: `{exc}`")

    check_output.object = "\n\n".join(lines)


check_button = pn.widgets.Button(name="Run checks for current frame", button_type="primary")
check_output = pn.pane.Markdown("Checks run on demand; the separate ERA5 `vo` file is downloaded only here.")
check_button.on_click(run_checks)

controls = pn.Column(
    "### Controls",
    diagnostic_widget,
    time_player,
    pn.pane.Markdown(pn.bind(lambda i: f"**ERA5 timestamp:** `{timestamp(i)}`", time_player.param.value)),
    spectral_widget,
    taper_enabled,
    taper_value,
    grid_widget,
    width=300,
)
plots = pn.Row(
    pn.Column("### Original", left_plot),
    pn.Column("### Processed", right_plot),
)
main = pn.Column(
    "# spharmgrid — ERA5 850-hPa spherical harmonic explorer",
    "Compare original and processed ERA5 wind on the same timestamp.",
    plots,
    summary_pane,
    pn.Card(check_button, check_output, title="Consistency checks"),
    pn.pane.Markdown("**Data:** ERA5, Copernicus Climate Change Service / ECMWF; distributed through PyStormTracker-Data `v0.2.0-data`."),
)
app = pn.Row(controls, main, sizing_mode="stretch_width")
app.servable(title="spharmgrid — ERA5 850-hPa spherical harmonic explorer")
app